# <b style='font-size:30px;font-family:Arial'>Create a File-Content-Based Collection using Parquet Files</b>

File-content-based vector collections are ideal for unstructured data where you provide files as the input source. The workflow is:
1. Configure the Ingestor pipeline with `.files()` and `.upsert()`
2. Execute the pipeline with `.run()` to create collection, ingest files, generate embeddings, and create index
3. Perform similarity searches

**Key Learning:** For FILE-CONTENT-BASED collections, use the method chain `.files().upsert().run()` where embeddings are generated from the text field during ingestion.

## <b style='font-size:20px;font-family:Arial'>1. Import Required Libraries</b>

In [ ]:
# Import Required Libraries
import sys
import os
import json
import glob
import getpass
from datetime import datetime
import pyarrow.parquet as pq

# TeradataML imports
from teradataml import create_context, remove_context, DataFrame

# Teradata Vector Store V2 API - Ingestor workflow
from teradatagenai import Collection, CollectionManager, ColumnInfo
from teradatagenai import BasicIngestor, ExtractionSchema
from teradatagenai import LocalConfig, S3Config, AzureBlobConfig
from teradatagenai import TeradataAI
from teradatagenai.vector_store.Ingestor import Ingestor
from teradatagenai.common.constants import CollectionType
from teradatasqlalchemy.types import VARCHAR, CLOB

print("✅ All imports successful!")
print("✓ Using Vector Store V2 API - Ingestor stepwise pipeline")
print("✓ Collection Type: FILE-CONTENT-BASED")

✅ All imports successful!
✓ Using Vector Store V2 API - Ingestor stepwise pipeline
✓ Collection Type: FILE-CONTENT-BASED


## <b style='font-size:20px;font-family:Arial'>2. Connect to Teradata Vantage</b>

In [ ]:
# Configure connection parameters
os.environ['TD_HOST'] = getpass.getpass('Enter Teradata Host: ')
os.environ['TD_USERNAME'] = getpass.getpass('Enter Teradata Username: ')
os.environ['TD_PASSWORD'] = getpass.getpass('Enter Teradata Password: ')
os.environ['TD_BASE_URL'] = getpass.getpass('Enter Base URL: ')
os.environ['TD_AUTH_MECH'] = 'BASIC'
os.environ['NVIDIA_API_KEY'] = getpass.getpass('Enter NVIDIA API Key: ')

# Create connection context
con = create_context()
print(f"✓ Connected to Teradata Vantage at {os.environ['TD_HOST']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✓ Vector Store service is healthy")
except Exception as e:
    print(f"Service status: {e}")

Authentication token is generated and set for the session.
✓ Connected to Teradata Vantage at 10.27.178.11
✓ Vector Store service is healthy


## <b style='font-size:20px;font-family:Arial'>3. Configure Embedding Model</b>

Specify the embedding model that will generate vector embeddings from the text content.

In [ ]:
# Configure embedding model (runtime generation)
os.environ["NVIDIA_API_KEY"] = getpass.getpass('Enter NVIDIA API Key: ')
embedding_model = TeradataAI(
    api_type="nim",
    model_name=getpass.getpass('Enter Embedding Model Name: '),
    api_base=getpass.getpass('Enter Embedding Model API Base URL: ')
)

print(f"✓ Embedding Model: {embedding_model.model_name}")
print("✓ Embeddings will be generated automatically during ingestion")

✓ Embedding Model: nvidia/llama-3.2-nv-embedqa-1b-v2
✓ Embeddings will be generated automatically during ingestion


## <b style='font-size:20px;font-family:Arial'>4. Prepare Parquet Input Data</b>

Load Parquet files for the vector collection. You can load from local storage, S3, or Azure Blob Storage.

## <b style='font-size:20px;font-family:Arial'>4a. Load from Local Storage</b>

Using Local files for ingestion

In [ ]:
# Use existing combined Parquet file  
notebook_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(notebook_dir), "example-data", "parquet_files")
parquet_file_path = os.path.join(data_dir, "combined_documents.parquet")

# Verify file exists
if not os.path.exists(parquet_file_path):
    raise FileNotFoundError(f"Combined Parquet file not found: {parquet_file_path}")

print(f"✓ Loading Parquet file: {os.path.basename(parquet_file_path)}")
print(f"✓ File location: {data_dir}")

# Verify the schema using arrow schema
parquet_file = pq.ParquetFile(parquet_file_path)
schema = parquet_file.schema_arrow
print(f"\n✓ Schema verification:")
print(f"  Columns: {schema.names}")
print(f"  Types: {[str(field.type) for field in schema]}")

# Configure local file source
local_config = LocalConfig(
    files=[parquet_file_path],
    files_type="parquet"
)
print(f"\n✓ LocalConfig created for local file ingestion")

✓ Loading Parquet file: combined_documents.parquet
✓ File location: ./sample_data/test_data_ingestor_parquet_content

✓ Schema verification:
  Columns: ['id', 'doc_title', 'text', 'category', 'author']
  Types: ['string', 'string', 'string', 'string', 'string']

✓ LocalConfig created for local file ingestion


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_local = f"parquet_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_local = ExtractionSchema(
    table_name=f"parquet_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor_local = (
    Ingestor(
        name=collection_name_local,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="vsdemo01",
        description="File-Content-Based collection from Parquet files"
    )
    .files(
        files=local_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_local
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_local}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(local_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_content_20260217_003000
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: LocalConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [6]:
# Check if collection already exists and destroy it
try:
    existing_collection_local = Collection(name=collection_name_local)
    if existing_collection_local.exists:
        print(f"⚠ Collection '{collection_name_local}' already exists. Destroying it first...")
        existing_collection_local.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)  # Wait for cleanup
except Exception as e:
    pass

# Execute ingestion (creates collection and ingests files)
print("Starting ingestion pipeline...")
result_local = ingestor_local.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/10

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_local = existing_collection_local.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

# Access the actual results
results_list = search_results_local.similar_objects
result_count = search_results_local.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Dr. Sarah Chen
   Category: Technology
   Text: Vector databases are specialized database systems designed to store and efficiently query high-dimensional vector embeddings. These embeddings represe...

2. Semantic Search Implementation
   Author: Prof. David Martinez
   Category: Technology
   Text: Semantic search uses vector embeddings to understand the meaning and context of search queries, going beyond simple keyword matching. By converting bo...

3. Retrieval-Augmented Generation Systems
   Author: Dr. Amanda Foster
   Category: AI & ML
   Text: Retrieval-Augmented Generation (RAG) combines the power of large language models with external knowledge retrieval. This architecture first retrieves ...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_local.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_local}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_content_20260217_003000_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_content_table_20260217003000,1,combined_documents.parquet,"-0.000656999996863306,-0.00892600044608116,0.00604599993675947,-0.0191799998283386,0.0372919999063015,-0.00764800002798438,0.0271300002932549,-0.016677999868989,0.0351260006427765,-0.00428400002419949,-0.0221410002559423,0.0329280011355877,-0.00350199989043176,0.00675999978557229,-0.01326800044626,-0.0152589995414019,0.0278629995882511,-0.0120090004056692,-0.0182949993759394,0.0370480008423328,0.0124970003962517,-0.0329280011355877,-0.0222779996693134,-0.013221999630332,0.0060809999704361,0.0156100001186132,-0.0250090006738901,0.0365599989891052,-0.0325619988143444,0.0177459996193647,0.00680899992585182,-0.0180820003151894,0.00366799999028444,0.0124890003353357,-0.00475300010293722,-0.00491000013425946,0.0140690002590418,-0.0033780001103878,0.0114059997722507,0.0017630000365898,0.0313110016286373,-0.0293429996818304,0.0252529997378588,-0.00122700002975762,-0.0148620000109076,-0.0287480000406504,0.0282589998096228,-0.00812499970197678,0.000835000013466924,-0.0125810001045465,0.00760299991816282,0.0099029997363","-0.000656899414025247,-0.0089246341958642,0.00604507466778159,-0.0191770642995834,0.0372862927615643,-0.00764682935550809,0.0271258484572172,-0.0166754480451345,0.0351206250488758,-0.00428334437310696,-0.0221376121044159,0.0329229608178139,-0.00350146391429007,0.00675896508619189,-0.0132659701630473,-0.0152566637843847,0.0278587359935045,-0.0120071619749069,-0.0182921998202801,0.0370423309504986,0.0124950874596834,-0.0329229608178139,-0.0222745891660452,-0.0132199758663774,0.00608006911352277,0.0156076112762094,-0.0250051729381084,0.0365544036030769,-0.0325570143759251,0.0177432838827372,0.00680795777589083,-0.0180792324244976,0.00366743863560259,0.0124870892614126,-0.00475227274000645,-0.00490924855694175,0.0140668470412493,-0.00337748299352825,0.0114042544737458,0.00176273018587381,0.0313062109053135,-0.0293385088443756,0.0252491347491741,-0.00122681225184351,-0.0148597257211804,-0.0287436004728079,0.0282546747475863,-0.00812375638633966,0.000834872189443558,-0.0125790741294622,0.00760183623060584,0.0099014"
vsdemo01,parquet_content_table_20260217003000,2,combined_documents.parquet,"0.00151199998799711,0.0200500003993511,0.0312189999967813,-0.00177800003439188,0.0178379993885756,0.0136259999126196,0.0281220003962517,-0.0174560006707907,0.0292509999126196,0.0268709994852543,-0.0136799998581409,-0.00596999982371926,-0.00930800009518862,0.0311890002340078,0.0387270003557205,-0.0142670003697276,0.0147550003603101,-0.0117720002308488,0.021484000608325,0.0210570003837347,-0.02734399959445,0.00788099970668554,-0.0460509993135929,-0.0167239997535944,-0.00552000012248755,0.00197799992747605,-0.00714100012555718,-0.00638200016692281,-0.0101239997893572,0.0135880000889301,0.0254360008984804,-0.00968900043517351,-0.0313719995319843,-0.027191000059247,0.0144809996709228,-0.0010430000256747,0.0183409992605448,0.00130700005684048,-0.0260620005428791,-0.00189099996350706,0.0647580027580261,-0.00458100019022822,-0.0209810007363558,0.00824000034481287,-0.0190120004117489,-0.0420840010046959,0.000538999971468002,0.00553900003433228,0.00250599998980761,0.0264740008860826,-0.014572000131011,0.025116000324487","0.00151223386637866,0.0200531017035246,0.0312238279730082,-0.00177827500738204,0.0178407579660416,0.0136281074956059,0.0281263496726751,-0.0174586996436119,0.0292555242776871,0.0268751550465822,-0.0136821158230305,-0.00597092323005199,-0.00930943991988897,0.0311938244849443,0.0387329906225204,-0.0142692066729069,0.0147572821006179,-0.0117738209664822,0.0214873235672712,0.0210602562874556,-0.0273482277989388,0.00788221880793571,-0.0460581220686436,-0.0167265869677067,-0.00552085367962718,0.00197830586694181,-0.00714210467413068,-0.0063829873688519,-0.0101255653426051,0.013590102083981,0.0254399348050356,-0.00969049893319607,-0.0313768498599529,-0.027195205911994,0.0144832395017147,-

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_local.destroy()
print(f"✓ Collection '{collection_name_local}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4b. Load from S3 Storage</b>

Instead of local files, you can ingest Parquet files directly from Amazon S3.

In [ ]:
from teradatagenai.vector_store import S3Config

# S3 Configuration
s3_config = S3Config(
    bucket=getpass.getpass('Enter S3 Bucket: '),
    key=getpass.getpass('Enter S3 Key: '),
    aws_access_key_id=getpass.getpass('Enter AWS Access Key ID: '),
    aws_secret_access_key=getpass.getpass('Enter AWS Secret Access Key: '),
    region_name=getpass.getpass('Enter AWS Region: '),
    files_type="parquet",
    aws_session_token=getpass.getpass('Enter AWS Session Token: ')
)

print("✓ S3 storage configuration enabled")
print(f"  Bucket: {s3_config.bucket}")
print(f"  Key: {s3_config.key}")
print(f"  Region: {s3_config.region_name}")

✓ S3 storage configuration enabled
  Bucket: genaitestaanchal
  Key: files/combined_documents.parquet
  Region: us-west-2


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_s3 = f"parquet_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_s3 = ExtractionSchema(
    table_name=f"parquet_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor_s3 = (
    Ingestor(
        name=collection_name_s3,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="vsdemo01",
        description="File-Content-Based collection from Parquet files"
    )
    .files(
        files=s3_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_s3
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_s3}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(s3_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_content_20260217_003327
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: S3Config
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [11]:
# Check if collection already exists and destroy it
try:
    existing_collection_s3 = Collection(name=collection_name_s3)
    if existing_collection_s3.exists:
        print(f"⚠ Collection '{collection_name_s3}' already exists. Destroying it first...")
        existing_collection_s3.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_s3 = ingestor_s3.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_content_20260217_003327' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_s3 = existing_collection_s3.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_s3.similar_objects
result_count = search_results_s3.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Dr. Sarah Chen
   Category: Technology
   Text: Vector databases are specialized database systems designed to store and efficiently query high-dimensional vector embeddings. These embeddings represe...

2. Semantic Search Implementation
   Author: Prof. David Martinez
   Category: Technology
   Text: Semantic search uses vector embeddings to understand the meaning and context of search queries, going beyond simple keyword matching. By converting bo...

3. Retrieval-Augmented Generation Systems
   Author: Dr. Amanda Foster
   Category: AI & ML
   Text: Retrieval-Augmented Generation (RAG) combines the power of large language models with external knowledge retrieval. This architecture first retrieves ...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_s3.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_s3}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_content_20260217_003327_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_content_table_20260217003327,1,combined_documents.parquet,"-0.000656999996863306,-0.00892600044608116,0.00604599993675947,-0.0191799998283386,0.0372919999063015,-0.00764800002798438,0.0271300002932549,-0.016677999868989,0.0351260006427765,-0.00428400002419949,-0.0221410002559423,0.0329280011355877,-0.00350199989043176,0.00675999978557229,-0.01326800044626,-0.0152589995414019,0.0278629995882511,-0.0120090004056692,-0.0182949993759394,0.0370480008423328,0.0124970003962517,-0.0329280011355877,-0.0222779996693134,-0.013221999630332,0.0060809999704361,0.0156100001186132,-0.0250090006738901,0.0365599989891052,-0.0325619988143444,0.0177459996193647,0.00680899992585182,-0.0180820003151894,0.00366799999028444,0.0124890003353357,-0.00475300010293722,-0.00491000013425946,0.0140690002590418,-0.0033780001103878,0.0114059997722507,0.0017630000365898,0.0313110016286373,-0.0293429996818304,0.0252529997378588,-0.00122700002975762,-0.0148620000109076,-0.0287480000406504,0.0282589998096228,-0.00812499970197678,0.000835000013466924,-0.0125810001045465,0.00760299991816282,0.0099029997363","-0.000656899414025247,-0.0089246341958642,0.00604507466778159,-0.0191770642995834,0.0372862927615643,-0.00764682935550809,0.0271258484572172,-0.0166754480451345,0.0351206250488758,-0.00428334437310696,-0.0221376121044159,0.0329229608178139,-0.00350146391429007,0.00675896508619189,-0.0132659701630473,-0.0152566637843847,0.0278587359935045,-0.0120071619749069,-0.0182921998202801,0.0370423309504986,0.0124950874596834,-0.0329229608178139,-0.0222745891660452,-0.0132199758663774,0.00608006911352277,0.0156076112762094,-0.0250051729381084,0.0365544036030769,-0.0325570143759251,0.0177432838827372,0.00680795777589083,-0.0180792324244976,0.00366743863560259,0.0124870892614126,-0.00475227274000645,-0.00490924855694175,0.0140668470412493,-0.00337748299352825,0.0114042544737458,0.00176273018587381,0.0313062109053135,-0.0293385088443756,0.0252491347491741,-0.00122681225184351,-0.0148597257211804,-0.0287436004728079,0.0282546747475863,-0.00812375638633966,0.000834872189443558,-0.0125790741294622,0.00760183623060584,0.0099014"
vsdemo01,parquet_content_table_20260217003327,6,combined_documents.parquet,"-0.0205380003899336,0.00680899992585182,0.00156999996397644,0.0220640003681183,0.00539000006392598,0.0203089993447065,0.0357359983026981,-0.00535999983549118,0.0357969999313354,0.00853000022470951,-0.00945299956947565,0.00797300040721893,0.00529900006949902,0.0152279995381832,0.000823999987915158,-0.0231630001217127,0.0229339990764856,-6.29999994998798e-05,-0.00291800010018051,0.0102770002558827,0.0104830004274845,-0.0682369992136955,-0.0148930000141263,-0.00839199963957071,-0.0070380000397563,0.0365599989891052,-0.00562300020828843,0.00405099987983704,0.0188139993697405,-0.0084840003401041,0.00123199995141476,-0.00776699976995587,-0.0228119995445013,0.00865899957716465,-0.0157469995319843,-0.0139009999111295,-0.00619899993762374,-0.0487059988081455,0.00106000003870577,-0.031555000692606,0.0242459997534752,0.0173649992793798,0.0195920001715422,-0.00593899982050061,-0.00307699991390109,-0.0338749997317791,-0.0111849997192621,0.0168760009109974,0.0197299998253584,-0.00236700009554625,0.0353090018033981,0.005797","-0.0205359365791082,0.00680831586942077,0.00156984222121537,0.0220617838203907,0.00538945849984884,0.0203069597482681,0.0357324071228504,-0.00535946153104305,0.0357934050261974,0.00852914340794086,-0.00945204962044954,0.00797219946980476,0.00529846781864762,0.0152264693751931,0.000823917216621339,-0.0231606736779213,0.022931694984436,-6.29936694167554e-05,-0.00291770696640015,0.0102759674191475,0.0104819471016526,-0.0682301446795464,-0.0148915033787489,-0.00839115679264069,-0.00703729270026088,0.0365563258528709,-0.00562243536114693,0.00405059289187193,0.018812108784914,-0.00848314817994833,0.00123187620192766,-0.00776621932163835,-0.0228097066283226,0.00865812972187996,-0.01574541

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_s3.destroy()
print(f"✓ Collection '{collection_name_s3}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4c. Load from Azure Blob Storage</b>

Instead of local files or S3, you can ingest Parquet files from Azure Blob Storage.

In [ ]:
from teradatagenai.vector_store import AzureBlobConfig

# Azure Blob Configuration
azure_config = AzureBlobConfig(
    container=getpass.getpass('Enter Azure Container: '),
    blob_name=getpass.getpass('Enter Blob Name: '),
    account_name=getpass.getpass('Enter Azure Account Name: '),
    account_key=getpass.getpass('Enter Azure Account Key: '),
    files_type="parquet"
)

print("✓ Azure Blob storage configuration provided")
print(f"  Container: {azure_config.container}")
print(f"  Blob: {azure_config.blob_name}")
print(f"  Account: {azure_config.account_name}")

✓ Azure Blob storage configuration provided
  Container: genaitestcontainer
  Blob: combined_documents.parquet
  Account: genaitestaanchal


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_blob = f"parquet_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_blob = ExtractionSchema(
    table_name=f"parquet_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor_blob = (
    Ingestor(
        name=collection_name_blob,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="vsdemo01",
        description="File-Content-Based collection from Parquet files"
    )
    .files(
        files=azure_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_blob
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_blob}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(azure_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_content_20260217_003536
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: AzureBlobConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [16]:
# Check if collection already exists and destroy it
try:
    existing_collection_blob = Collection(name=collection_name_blob)
    if existing_collection_blob.exists:
        print(f"⚠ Collection '{collection_name_blob}' already exists. Destroying it first...")
        existing_collection_blob.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_blob = ingestor_blob.run()
print("✓ Ingestion completed - files uploaded to collection")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_content_20260217_003536' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_blob = existing_collection_blob.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_blob.similar_objects
result_count = search_results_blob.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Dr. Sarah Chen
   Category: Technology
   Text: Vector databases are specialized database systems designed to store and efficiently query high-dimensional vector embeddings. These embeddings represe...

2. Semantic Search Implementation
   Author: Prof. David Martinez
   Category: Technology
   Text: Semantic search uses vector embeddings to understand the meaning and context of search queries, going beyond simple keyword matching. By converting bo...

3. Retrieval-Augmented Generation Systems
   Author: Dr. Amanda Foster
   Category: AI & ML
   Text: Retrieval-Augmented Generation (RAG) combines the power of large language models with external knowledge retrieval. This architecture first retrieves ...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_blob.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_blob}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_content_20260217_003536_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_content_table_20260217003536,1,combined_documents.parquet,"-0.000656999996863306,-0.00892600044608116,0.00604599993675947,-0.0191799998283386,0.0372919999063015,-0.00764800002798438,0.0271300002932549,-0.016677999868989,0.0351260006427765,-0.00428400002419949,-0.0221410002559423,0.0329280011355877,-0.00350199989043176,0.00675999978557229,-0.01326800044626,-0.0152589995414019,0.0278629995882511,-0.0120090004056692,-0.0182949993759394,0.0370480008423328,0.0124970003962517,-0.0329280011355877,-0.0222779996693134,-0.013221999630332,0.0060809999704361,0.0156100001186132,-0.0250090006738901,0.0365599989891052,-0.0325619988143444,0.0177459996193647,0.00680899992585182,-0.0180820003151894,0.00366799999028444,0.0124890003353357,-0.00475300010293722,-0.00491000013425946,0.0140690002590418,-0.0033780001103878,0.0114059997722507,0.0017630000365898,0.0313110016286373,-0.0293429996818304,0.0252529997378588,-0.00122700002975762,-0.0148620000109076,-0.0287480000406504,0.0282589998096228,-0.00812499970197678,0.000835000013466924,-0.0125810001045465,0.00760299991816282,0.0099029997363","-0.000656899414025247,-0.0089246341958642,0.00604507466778159,-0.0191770642995834,0.0372862927615643,-0.00764682935550809,0.0271258484572172,-0.0166754480451345,0.0351206250488758,-0.00428334437310696,-0.0221376121044159,0.0329229608178139,-0.00350146391429007,0.00675896508619189,-0.0132659701630473,-0.0152566637843847,0.0278587359935045,-0.0120071619749069,-0.0182921998202801,0.0370423309504986,0.0124950874596834,-0.0329229608178139,-0.0222745891660452,-0.0132199758663774,0.00608006911352277,0.0156076112762094,-0.0250051729381084,0.0365544036030769,-0.0325570143759251,0.0177432838827372,0.00680795777589083,-0.0180792324244976,0.00366743863560259,0.0124870892614126,-0.00475227274000645,-0.00490924855694175,0.0140668470412493,-0.00337748299352825,0.0114042544737458,0.00176273018587381,0.0313062109053135,-0.0293385088443756,0.0252491347491741,-0.00122681225184351,-0.0148597257211804,-0.0287436004728079,0.0282546747475863,-0.00812375638633966,0.000834872189443558,-0.0125790741294622,0.00760183623060584,0.0099014"
vsdemo01,parquet_content_table_20260217003536,3,combined_documents.parquet,"-0.00479099992662668,0.0126649998128414,0.0273589994758368,-0.00276600010693073,0.0118410000577569,0.0538330003619194,0.0140840001404285,-0.0147320004180074,0.0280760005116463,0.0308839995414019,-0.0360720008611679,-0.0227359998971224,-0.0353390015661716,0.045134998857975,-0.0120700001716614,-5.40000000910368e-05,0.00584000023081899,0.0303959995508194,-0.0142820002511144,0.033661000430584,-0.000651000009384006,-0.00729000009596348,-0.00143099995329976,-0.0203089993447065,0.024795999750495,-0.0209960006177425,0.0187229998409748,-0.00827800016850233,-0.00495100021362305,-0.0120320003479719,0.00931499991565943,-0.00884999986737967,0.016907000914216,-0.00272600003518164,0.0107039995491505,-0.0142970001325011,0.0160680003464222,0.00109100004192442,-0.0123829999938607,-0.0468139983713627,0.0449829995632172,0.00196600006893277,0.0198359992355108,0.0110400002449751,-0.0151209998875856,-0.0253139995038509,0.0120620001107454,0.0176999997347593,-0.0170440003275871,-0.0141070000827312,-0.00814099982380867,0.0110240001231","-0.00478992937132716,0.0126621704548597,0.0273528881371021,-0.00276538217440248,0.011838355101645,0.053820975124836,0.0140808541327715,-0.0147287091240287,0.0280697289854288,0.0308771003037691,-0.0360639430582523,-0.0227309204638004,-0.0353311076760292,0.0451249144971371,-0.0120673039928079,-5.39879365533125e-05,0.00583869544789195,0.0303892083466053,-0.0142788095399737,0.0336534790694714,-0.000650854548439384,-0.00728837167844176,-0.00143068027682602,-0.020304461941123,0.0247904602438211,-0.0209913104772568,0.0187188163399696,-0.00827615056186914,-0.00494989426806569,-0.0120293125510216,0.00931291934102774,-0.00884802266955376,0.0169032234698534,-0.00272539095021784,0.010701607912

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_blob.destroy()
print(f"✓ Collection '{collection_name_blob}' destroyed")

## <b style='font-size:20px;font-family:Arial'>4d. Load from Google Cloud Storage</b>

Instead of local files, S3, or Azure, you can ingest Parquet files directly from Google Cloud Platform (GCP) Storage.

In [ ]:
from teradatagenai import GCPConfig

# GCP Configuration
gcp_config = GCPConfig(
    files_type="parquet",
    bucket=getpass.getpass('Enter GCP Bucket: '),
    blob_name=getpass.getpass('Enter GCP Blob Name: '),
    project_id=getpass.getpass('Enter GCP Project ID: '),
    secret=json.loads(getpass.getpass('Enter GCP Service Account JSON (paste entire JSON): '))
)

print("✓ GCP storage configuration enabled")
print(f"  Bucket: {gcp_config.bucket}")
print(f"  Blob: {gcp_config.blob_name}")
print(f"  Project: {gcp_config.project_id}")

✓ GCP storage configuration enabled
  Bucket: ak250090-filestore
  Blob: harsha_files/combined_documents.parquet
  Project: icgcp-vector-store


Run Ingestion Pipeline and Query Results

In [ ]:
# Generate unique collection name
collection_name_gcp = f"parquet_file_content_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Define extraction schema
extraction_schema_gcp = ExtractionSchema(
    table_name=f"parquet_content_table_{datetime.now().strftime('%Y%m%d%H%M%S')}",
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(32000))
    ],
    metadata_columns=[
        ColumnInfo(name="id", datatype=VARCHAR(50)),
        ColumnInfo(name="doc_title", datatype=VARCHAR(500)),
        ColumnInfo(name="category", datatype=VARCHAR(100)),
        ColumnInfo(name="author", datatype=VARCHAR(200))
    ]
)

# Build Ingestor pipeline
ingestor_gcp = (
    Ingestor(
        name=collection_name_gcp,
        type=CollectionType.FILE_CONTENT_BASED,
        target_database="vsdemo01",
        description="File-Content-Based collection from Parquet files with runtime embedding generation"
    )
    .files(
        files=gcp_config,
        ingestor=BasicIngestor(chunk_size=512, chunk_overlap=50),
        extraction_schema=extraction_schema_gcp
    )
    .upsert(embedding_model=embedding_model)
)

print(f"✓ Collection Name: {collection_name_gcp}")
print("✓ Collection Type: FILE-CONTENT-BASED")
print(f"✓ Storage: {type(gcp_config).__name__}")
print("✓ Ingestor pipeline configured")

✓ Collection Name: parquet_file_content_20260217_003746
✓ Collection Type: FILE-CONTENT-BASED
✓ Storage: GCPConfig
✓ Ingestor pipeline configured


Check for an existing collection, destroys it if present, and then runs the ingestion pipeline to upload files into the collection.

In [21]:
# Check if collection already exists and destroy it
try:
    existing_collection_gcp = Collection(name=collection_name_gcp)
    if existing_collection_gcp.exists:
        print(f"⚠ Collection '{collection_name_gcp}' already exists. Destroying it first...")
        existing_collection_gcp.destroy()
        print("✓ Existing collection destroyed\n")
        import time
        time.sleep(2)
except Exception as e:
    pass

# Execute ingestion
print("Starting ingestion pipeline...")
result_gcp = ingestor_gcp.run()
print("✓ Ingestion completed - embeddings generated and stored")

c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Starting ingestion pipeline...
Initializing collection pipeline...


c:\Users\sp255175\Documents\GIT-REPO-TD\NexusAI\test_venv\Lib\site-packages\teradatagenai\vector_store\collection.py:560: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'parquet_file_content_20260217_003746' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

Perform similarity search and access the actual results from the _SimilaritySearch object

In [ ]:
# Perform similarity search
search_results_gcp = existing_collection_gcp.similarity_search(
    question="What are vector databases used for?",
    top_k=3,
    return_type="json"
)

results_list = search_results_gcp.similar_objects
result_count = search_results_gcp.similar_objects_count

print(f"\nSearch Results ({result_count} documents):")
print("="*80)
for i, result in enumerate(results_list, 1):
    print(f"\n{i}. {result.get('doc_title', 'N/A')}")
    print(f"   Author: {result.get('author', 'N/A')}")
    print(f"   Category: {result.get('category', 'N/A')}")
    print(f"   Text: {result.get('text', '')[:150]}...")


Search Results (3 documents):

1. Introduction to Vector Databases
   Author: Dr. Sarah Chen
   Category: Technology
   Text: Vector databases are specialized database systems designed to store and efficiently query high-dimensional vector embeddings. These embeddings represe...

2. Semantic Search Implementation
   Author: Prof. David Martinez
   Category: Technology
   Text: Semantic search uses vector embeddings to understand the meaning and context of search queries, going beyond simple keyword matching. By converting bo...

3. Retrieval-Augmented Generation Systems
   Author: Dr. Amanda Foster
   Category: AI & ML
   Text: Retrieval-Augmented Generation (RAG) combines the power of large language models with external knowledge retrieval. This architecture first retrieves ...


View ingested data

In [ ]:
# View ingested data using Collection method
try:
    indexes_df = existing_collection_gcp.get_indexes_embeddings()
    print(f"\n✓ Sample data from collection '{collection_name_gcp}':")
    display(indexes_df)
except Exception as e:
    print(f"Note: {e}")


Table: collection_parquet_file_content_20260217_003746_index


DataBaseName,TableName,TD_ID,td_filename,vector_index,vector_index_normalized
vsdemo01,parquet_content_table_20260217003746,1,combined_documents.parquet,"-0.000656999996863306,-0.00892600044608116,0.00604599993675947,-0.0191799998283386,0.0372919999063015,-0.00764800002798438,0.0271300002932549,-0.016677999868989,0.0351260006427765,-0.00428400002419949,-0.0221410002559423,0.0329280011355877,-0.00350199989043176,0.00675999978557229,-0.01326800044626,-0.0152589995414019,0.0278629995882511,-0.0120090004056692,-0.0182949993759394,0.0370480008423328,0.0124970003962517,-0.0329280011355877,-0.0222779996693134,-0.013221999630332,0.0060809999704361,0.0156100001186132,-0.0250090006738901,0.0365599989891052,-0.0325619988143444,0.0177459996193647,0.00680899992585182,-0.0180820003151894,0.00366799999028444,0.0124890003353357,-0.00475300010293722,-0.00491000013425946,0.0140690002590418,-0.0033780001103878,0.0114059997722507,0.0017630000365898,0.0313110016286373,-0.0293429996818304,0.0252529997378588,-0.00122700002975762,-0.0148620000109076,-0.0287480000406504,0.0282589998096228,-0.00812499970197678,0.000835000013466924,-0.0125810001045465,0.00760299991816282,0.0099029997363","-0.000656899414025247,-0.0089246341958642,0.00604507466778159,-0.0191770642995834,0.0372862927615643,-0.00764682935550809,0.0271258484572172,-0.0166754480451345,0.0351206250488758,-0.00428334437310696,-0.0221376121044159,0.0329229608178139,-0.00350146391429007,0.00675896508619189,-0.0132659701630473,-0.0152566637843847,0.0278587359935045,-0.0120071619749069,-0.0182921998202801,0.0370423309504986,0.0124950874596834,-0.0329229608178139,-0.0222745891660452,-0.0132199758663774,0.00608006911352277,0.0156076112762094,-0.0250051729381084,0.0365544036030769,-0.0325570143759251,0.0177432838827372,0.00680795777589083,-0.0180792324244976,0.00366743863560259,0.0124870892614126,-0.00475227274000645,-0.00490924855694175,0.0140668470412493,-0.00337748299352825,0.0114042544737458,0.00176273018587381,0.0313062109053135,-0.0293385088443756,0.0252491347491741,-0.00122681225184351,-0.0148597257211804,-0.0287436004728079,0.0282546747475863,-0.00812375638633966,0.000834872189443558,-0.0125790741294622,0.00760183623060584,0.0099014"
vsdemo01,parquet_content_table_20260217003746,2,combined_documents.parquet,"0.00151199998799711,0.0200500003993511,0.0312189999967813,-0.00177800003439188,0.0178379993885756,0.0136259999126196,0.0281220003962517,-0.0174560006707907,0.0292509999126196,0.0268709994852543,-0.0136799998581409,-0.00596999982371926,-0.00930800009518862,0.0311890002340078,0.0387270003557205,-0.0142670003697276,0.0147550003603101,-0.0117720002308488,0.021484000608325,0.0210570003837347,-0.02734399959445,0.00788099970668554,-0.0460509993135929,-0.0167239997535944,-0.00552000012248755,0.00197799992747605,-0.00714100012555718,-0.00638200016692281,-0.0101239997893572,0.0135880000889301,0.0254360008984804,-0.00968900043517351,-0.0313719995319843,-0.027191000059247,0.0144809996709228,-0.0010430000256747,0.0183409992605448,0.00130700005684048,-0.0260620005428791,-0.00189099996350706,0.0647580027580261,-0.00458100019022822,-0.0209810007363558,0.00824000034481287,-0.0190120004117489,-0.0420840010046959,0.000538999971468002,0.00553900003433228,0.00250599998980761,0.0264740008860826,-0.014572000131011,0.025116000324487","0.00151223386637866,0.0200531017035246,0.0312238279730082,-0.00177827500738204,0.0178407579660416,0.0136281074956059,0.0281263496726751,-0.0174586996436119,0.0292555242776871,0.0268751550465822,-0.0136821158230305,-0.00597092323005199,-0.00930943991988897,0.0311938244849443,0.0387329906225204,-0.0142692066729069,0.0147572821006179,-0.0117738209664822,0.0214873235672712,0.0210602562874556,-0.0273482277989388,0.00788221880793571,-0.0460581220686436,-0.0167265869677067,-0.00552085367962718,0.00197830586694181,-0.00714210467413068,-0.0063829873688519,-0.0101255653426051,0.013590102083981,0.0254399348050356,-0.00969049893319607,-0.0313768498599529,-0.027195205911994,0.0144832395017147,-

Destroy the collection

In [ ]:
# Clean up: Destroy the collection (comment out if you want to keep it)
existing_collection_gcp.destroy()
print(f"✓ Collection '{collection_name_gcp}' destroyed")

---

## Summary

This notebook demonstrated how to:
- Create a FILE-CONTENT-BASED vector collection from Parquet files
- Configure the Ingestor pipeline with extraction schema and embedding model
- Test with four different storage types: Local, S3, Azure Blob, and GCP
- Generate embeddings automatically during ingestion
- Perform semantic similarity search on the collection

**Key Points:**
- Parquet files are efficiently processed for vector storage
- Embeddings are generated automatically from the `content` field using the specified embedding model
- Metadata columns are preserved for filtering and retrieval
- Each storage type (Local/S3/Azure/GCP) has its own complete isolated workflow
- The collection supports semantic search out of the box